# Modelo Baseline: Buy and Hold

Buy and Hold es la predicción más simple posible: el promedio futuro de cada 
activo será igual a su último precio observado en la ventana de entrada. No 
hay entrenamiento, no hay parámetros, no hay aprendizaje.

Formalmente, dado `X` con forma `(n_muestras, ventana_entrada, n_activos)`, 
la predicción es `X[:, -1, :]`, es decir, el último día de cada ventana.


El enunciado pide explícitamente añadir modelos simples como referencia 
. Su utilidad es servir como **cota inferior de honestidad**: cualquier modelo 
neuronal que no supere este MAE no está aprendiendo nada útil de los datos, 
ya que la predicción trivial "mañana = hoy" lo iguala.

## Imports y carga de datos

**Returns en lugar de precios.** Trabajamos con rendimientos diarios 
  (`returns = precios.pct_change()`), siguiendo la práctica habitual en 
  finanzas cuantitativas y el enfoque del notebook de compartido en clase. Esto produce series aproximadamente estacionarias y MAE en 
  escala porcentual, comparables entre activos.

In [6]:
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
import os
from sklearn.model_selection import train_test_split

warnings.simplefilter(action="ignore", category=FutureWarning)

start_date = '1945-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD',
                   'GE', 'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM',
                   'MO', 'MRK', 'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date,
                            auto_adjust=True, progress=False)['Close']
precios_close.dropna(axis=1, inplace=True)

# Convertir a returns (rendimientos diarios)
returns = precios_close.pct_change().dropna()

print(f"Precios: {precios_close.shape}")
print(f"Returns: {returns.shape}")
print(f"Estadísticas de returns:\n{returns.describe().loc[['mean', 'std', 'min', 'max']].T.head()}")

Precios: (16192, 23)
Returns: (16191, 23)
Estadísticas de returns:
            mean       std       min       max
Ticker                                        
AEP     0.000395  0.013074 -0.227848  0.198417
BA      0.000665  0.021420 -0.238484  0.243186
CAT     0.000640  0.018654 -0.216216  0.147230
CNP     0.000458  0.017203 -0.422092  0.629630
CVX     0.000528  0.016127 -0.221248  0.227407


## Función para crear ventanas

In [7]:
def create_time_series_data(data, input_window_size, output_window_size):
    """
    Genera ventanas X (input_window_size días de entrada) e y (promedio
    de los output_window_size días siguientes) para cada activo.
    """
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    
    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_seq = data_array[i : i + input_window_size]
        output_seq = data_array[i + input_window_size : i + input_window_size + output_window_size]
        X.append(input_seq)
        y.append(np.mean(output_seq, axis=0))
    
    return np.array(X), np.array(y)

## Función Buy and Hold + cálculo del MAE

In [8]:
def predict_buy_and_hold(X):
    """
    Predicción: el promedio futuro = último precio observado de la ventana.
    X tiene forma (n_muestras, ventana_entrada, n_activos).
    Devuelve (n_muestras, n_activos).
    """
    return X[:, -1, :]   # último día de cada ventana, para todos los activos


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

## Loop sobre las 16 combinaciones

In [9]:
input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]
RANDOM_SEED = 42

resultados = []

for in_w in input_windows:
    for out_w in output_windows:
        # Generar ventanas sobre returns
        X, y = create_time_series_data(returns, in_w, out_w)
        
        # Split 1: 90/10 train+val / test (sin shuffle, cronológico)
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y, test_size=0.1, shuffle=False, random_state=RANDOM_SEED
        )
        
        # Split 2: del train sacamos 5% para validación
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_full, y_train_full, test_size=0.05, shuffle=True, 
            random_state=RANDOM_SEED
        )
        
        # Predicciones Buy and Hold
        y_pred_train = predict_buy_and_hold(X_train)
        y_pred_val   = predict_buy_and_hold(X_val)
        y_pred_test  = predict_buy_and_hold(X_test)
        
        resultados.append({
            'input_window': in_w,
            'output_window': out_w,
            'mae_train': mae(y_train, y_pred_train),
            'mae_val':   mae(y_val,   y_pred_val),
            'mae_test':  mae(y_test,  y_pred_test),
        })

df_resultados = pd.DataFrame(resultados)
df_resultados

,input_window,output_window,mae_train,mae_val,mae_test
0,5,1,0.016354,0.016469,0.017786
1,5,5,0.012941,0.013217,0.013659
2,5,30,0.011850,0.011835,0.012518
3,5,90,0.011675,0.011751,0.012248
4,10,1,0.016355,0.016480,0.017786
5,10,5,0.012948,0.013089,0.013662
6,10,30,0.011849,0.011867,0.012518
7,10,90,0.011678,0.011695,0.012248
8,30,1,0.016378,0.016105,0.017793
9,30,5,0.012961,0.012896,0.013667


## Guardar resultados

In [10]:
import os
os.makedirs('../results', exist_ok=True)
df_resultados.to_csv('../results/buy_and_hold_resultados.csv', index=False)
print("Resultados guardados en results/buy_and_hold_resultados.csv")

Resultados guardados en results/buy_and_hold_resultados.csv


## Conclusiones

El baseline Buy and Hold establece la cota de referencia para las 16 
combinaciones de ventanas (entrada × salida) que pide el enunciado. Las 
observaciones principales:

- **MAE de train, validación y test prácticamente idénticos.** Las tres 
  particiones presentan valores muy similares en todas las combinaciones 
  (por ejemplo, en la combinación 90/90: 0.0117 / 0.0114 / 0.0122). Esto 
  confirma que, al usar particiones aleatorias sobre returns estacionarios, 
  los tres conjuntos comparten distribución. Como además este modelo no se 
  entrena, no puede haber overfitting y las pequeñas diferencias se deben 
  únicamente a varianza muestral.

- **El MAE DECRECE con la ventana de salida.** En test pasa de ~0.0178 
  (salida 1 día) a ~0.0122 (salida 90 días). Es contraintuitivo a primera 
  vista, pero coherente con la naturaleza de los returns: al promediar el 
  return de muchos días, el ruido se cancela y el promedio tiende a cero 
  (los returns diarios tienen media muy próxima a cero a largo plazo). 
  Como Buy and Hold predice "el último return observado", que también es 
  pequeño en magnitud, el error absoluto se reduce. Para los modelos 
  neuronales esto implica que las ventanas de salida cortas (1, 5 días) 
  serán las más exigentes, mientras que las largas (30, 90 días) ofrecen 
  un baseline más fácil de batir aparentemente, pero también una señal 
  más débil que aprender.

- **El MAE no depende de la ventana de entrada.** Para una misma ventana 
  de salida, los resultados son prácticamente idénticos entre 5, 10, 30 y 
  90 días de entrada (ej. para salida 1: 0.01779, 0.01779, 0.01779, 
  0.01783). Es lo esperado: este modelo solo utiliza el último día 
  observado e ignora todo el histórico anterior. Los modelos neuronales 
  sí deberían beneficiarse de tener más contexto temporal.

Estos resultados servirán como referencia mínima a batir en los siguientes 
notebooks (regresión lineal, MLP, RNN, CNN y modelos mixtos).